In [1]:
#conversacion con gemini
# https://gemini.google.com/app/2622fa6806eb26a5

import os
import subprocess
try:
    _printenv = subprocess.run(
        ['bash', '-c', 'source ~/.bashrc 2>/dev/null && printenv'],
        text=True, capture_output=True, timeout=10,
    ).stdout
    for _line in _printenv.splitlines():
        if '=' in _line:
            _k, _v = _line.split('=', 1)
            os.environ.setdefault(_k, _v)
except Exception:
    pass
if 'PDK_ROOT' in os.environ and 'PDK' in os.environ:
    os.environ.setdefault('PDKPATH', os.path.join(os.environ['PDK_ROOT'], os.environ['PDK']))

In [2]:
#Variables de entorno dentro del contenedor del docker
print(_printenv)

SHELL=/bin/bash
SESSION_MANAGER=local/dbc44adcdc7b:@/tmp/.ICE-unix/59,unix/dbc44adcdc7b:/tmp/.ICE-unix/59
OMPI_MCA_btl_vader_single_copy_mechanism=none
PDK=gf180mcuD
COLORTERM=truecolor
XDG_CONFIG_DIRS=/etc/xdg
XDG_MENU_PREFIX=xfce-
NSS_WRAPPER_GROUP=/tmp/group
CONDA_EXE=/headless/conda-env/miniconda3/bin/conda
_CE_M=
KLAYOUT_PATH=/headless/.klayout:/foss/pdks/gf180mcuD/libs.tech/klayout
HOSTNAME=dbc44adcdc7b
SSH_AUTH_SOCK=/tmp/ssh-P7Kxh7Aj5tmo/agent.89
XDG_DATA_HOME=/headless/.data-default
NO_VNC_HOME=/usr/share/novnc
NSS_WRAPPER_PASSWD=/tmp/passwd
DESKTOP_SESSION=xfce
SSH_AGENT_PID=90
NO_AT_BRIDGE=1
KLAYOUT_PYTHONPATH=/foss/tools/klayout_gdsfactory9/lib/python3.12/site-packages
XML_CATALOG_FILES=file:///headless/conda-env/miniconda3/envs/GLdev/etc/xml/catalog file:///etc/xml/catalog
EDITOR=gedit
PWD=/foss/designs
VNC_RESOLUTION=1680x1050
GSETTINGS_SCHEMA_DIR=/headless/conda-env/miniconda3/envs/GLdev/share/glib-2.0/schemas
XDG_SESSION_TYPE=x11
CONDA_PREFIX=/headless/conda-env/minicond

In [2]:
#funciones para ir desplegando el gds y desplegar los componentes en jupiter

import gdstk
import svgutils.transform as sg
import IPython.display
from IPython.display import clear_output
import ipywidgets as widgets

# Redirect all outputs here
hide = widgets.Output()

def display_gds(gds_file,path,scale = 3):
  
  # Generate an SVG image
  top_level_cell = gdstk.read_gds(gds_file).top_level()[0]
  top_level_cell.write_svg(os.path.join(path,'out.svg'))
    
  # Scale the image for displaying
  fig = sg.fromfile(os.path.join(path,'out.svg'))
  fig.set_size((str(float(fig.width) * scale), str(float(fig.height) * scale)))
  fig.save(os.path.join(path,'out.svg'))

  # Display the image
  IPython.display.display(IPython.display.SVG(os.path.join(path,'out.svg')))
  os.remove(os.path.join(path,'out.gds'))

def display_component(component,path,scale = 3):
  # Save to a GDS file
  with hide:
    component.write_gds(os.path.join(path,'out.gds'))
  display_gds(os.path.join(path,'out.gds'),path,scale)

In [3]:
# %%

import os
import gdsfactory as gf #duda
from gdsfactory import Component
from glayout import MappedPDK, gf180
from gdsfactory.components import text_freetype, rectangle
from glayout import nmos, pmos
from glayout.primitives.mimcap import mimcap, mimcap_array #Genera un capacitor tipo MIM (Metal-Insulator-Metal)
from glayout.routing.straight_route import straight_route
from glayout.routing.c_route import c_route
from glayout.routing.L_route import L_route
from glayout.util.comp_utils import align_comp_to_port, evaluate_bbox, prec_center, prec_ref_center
from glayout.util.port_utils import add_ports_perimeter, print_ports
from glayout.util.snap_to_grid import component_snap_to_grid
from glayout.spice.netlist import Netlist
from glayout import via_stack
from glayout import rename_ports_by_orientation
from glayout import tapring

In [4]:


integrator_config = {
    "pdk": gf180,
    "layout_rules": {
        "spacing": gf180.util_max_metal_seperation(),
        "routing_metal": "met2",
        "dummy_devices": True,
        "tie_layers": ("met2", "met1"),
        "sd_rmult": 1,
    },
}

# Base parameters for PMOS and NMOS
nmos_kwargs = {
    "with_tie": True, #checar a true
    "with_dnwell": False,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2", "met1"),
    "dummy_routes": False,
}

pmos_kwargs = {
    "with_tie": True,#chacar a True
    "dnwell": False,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2", "met1"),
    "dummy_routes": False,
}

In [ ]:
# .subckt integrator_final_mike Vext_pin avdd avss I_50n
# *.PININFO Vext_pin:I avdd:B avss:B I_50n:I
# XM1 vg vg vm avdd pfet_03v3 L=.28u W=1u nf=1 ad='int((nf+1)/2) * W/nf * 0.18u' as='int((nf+2)/2) * W/nf * 0.18u' pd='2*int((nf+1)/2) * (W/nf + 0.18u)'
# + ps='2*int((nf+2)/2) * (W/nf + 0.18u)' nrd='0.18u / W' nrs='0.18u / W' sa=0 sb=0 sd=0
# XM6 vm Vext_pin avdd avss nfet_03v3 L=0.28u W=1u nf=1 ad='int((nf+1)/2) * W/nf * 0.18u' as='int((nf+2)/2) * W/nf * 0.18u' pd='2*int((nf+1)/2) * (W/nf + 0.18u)'
# + ps='2*int((nf+2)/2) * (W/nf + 0.18u)' nrd='0.18u / W' nrs='0.18u / W' sa=0 sb=0 sd=0
# XM2 avss I_50n vg avss nfet_03v3 L=0.28u W=1u nf=1 ad='int((nf+1)/2) * W/nf * 0.18u' as='int((nf+2)/2) * W/nf * 0.18u' pd='2*int((nf+1)/2) * (W/nf + 0.18u)'
# + ps='2*int((nf+2)/2) * (W/nf + 0.18u)' nrd='0.18u / W' nrs='0.18u / W' sa=0 sb=0 sd=0
# XM3 I_50n I_50n avss avss nfet_03v3 L=0.28u W=1u nf=1 ad='int((nf+1)/2) * W/nf * 0.18u' as='int((nf+2)/2) * W/nf * 0.18u' pd='2*int((nf+1)/2) * (W/nf + 0.18u)'
# + ps='2*int((nf+2)/2) * (W/nf + 0.18u)' nrd='0.18u / W' nrs='0.18u / W' sa=0 sb=0 sd=0
# XC3 vm avss cap_mim_2f0fF c_width=17.68e-6 c_length=17.68e-6 m=8
# .ends


In [5]:
# ----------------------------------------------------------------------
# 2. Instantiate Devices
# ----------------------------------------------------------------------
pdk = integrator_config["pdk"]
integrator = Component(name="integrator")

# PMOS Transistors (XM1, XM6)
xm1 = pmos(pdk, width=1.0, length=0.28, fingers=1,  with_dummy=(False, False), with_substrate_tap=False, **pmos_kwargs)


# NMOS Transistors (XM2, XM3)
xm2 = nmos(pdk, width=1.0, length=0.28, fingers=1, with_dummy=(False, False), with_substrate_tap=False,  **nmos_kwargs)
xm3 = nmos(pdk, width=1.0, length=0.28, fingers=1,  with_dummy=(False, False), with_substrate_tap=False, **nmos_kwargs)
xm6 = nmos(pdk, width=1.0, length=0.28, fingers=1,  with_dummy=(False, False), with_substrate_tap=False, **nmos_kwargs)


# MIM Capacitor Array XC3 (m=8, 2 rows x 4 columns)
xc3 = mimcap_array(
    pdk, 
    size=(17.68, 17.68), 
    rows=2, 
    columns=4
)


# Add components as references to the main cell
xm1_ref = integrator << xm1
xm6_ref = integrator << xm6
xm2_ref = integrator << xm2
xm3_ref = integrator << xm3
xc3_ref = integrator << xc3





xm1_ref.name ="xm1"
xm6_ref.name ="xm6"
xm2_ref.name ="xm2"
xm3_ref.name ="xm3"
xc3_ref.name ="xc3"





xm1_xy = evaluate_bbox(xm1)
xm2_xy = evaluate_bbox(xm2)
xm3_xy = evaluate_bbox(xm3)
xm6_xy = evaluate_bbox(xm6)
xc3_xy = evaluate_bbox(xc3)


# ----------------------------------------------------------------------
# 3. Component Placement & Floorplanning
# ----------------------------------------------------------------------
# Row 1 (Top): PMOS transistors side-by-side
# xm1_ref.move((15, 24))
sepa=5
xm6_ref.movex(xm6_xy[0]/2)
xm3_ref.movex(xm6_ref.xmax + xm3_xy[0]/2 + pdk.util_max_metal_seperation() + sepa)
xm2_ref.movex(xm3_ref.xmax + xm2_xy[0]/2 + pdk.util_max_metal_seperation() + sepa)
xc3_ref.movex(xm2_ref.xmax + xc3_xy[0]/2 - 13*2)
# Row 2 (Middle): Big MIM Capacitor Array
# xc3_ref.move((0, 0))

xm6_ref.movey(xm6_xy[1]/2)
xm2_ref.movey(xm2_xy[1]/2)
xm3_ref.movey(xm3_xy[1]/2)
xc3_ref.movey(xc3_xy[1]/2 - 12)

xm_max = max(xm6_xy[1], xm2_xy[1], xm3_xy[1])
separacion = 5


xm1_ref.movex(xm3_ref.xmax + 3)
xm1_ref.movey(xm_max + separacion + xm1_xy[1]/2)

# Row 3 (Bottom): NMOS transistors side-by-side
# xm6_ref.move((0, -4))
# xm2_ref.move((30, -4))
# xm3_ref.move((15, -4))
#return integrator_comp
# ----------------------------------------------------------------------


# ----------------------------------------------------------------------
# 5. Add Global Pins / Ports and Export GDS
# ----------------------------------------------------------------------
integrator.add_port("Vext_pin", port=xm6_ref.ports["multiplier_0_gate_E"])
integrator.add_port("I_50n", port=xm3_ref.ports["multiplier_0_gate_E"])
#integrator.add_port("avdd", port=xm6_ref.ports["multiplier_0_drain_W"])
#integrator.add_port("avss", port=xm2_ref.ports["multiplier_0_source_E"])



2026-08-25 17:17:55.767 | INFO     | gdsfactory.pdk:activate:337 - 'gf180' PDK is now active


{'name': 'I_50n', 'width': 0.5, 'center': [12.21, 1.355], 'orientation': 0.0, 'layer': [36, 0], 'port_type': 'electrical'}

In [6]:
# 4. Routing Net Connections
# ----------------------------------------------------------------------


viam2m3 = via_stack(pdk, "met2", "met3", centered=True)
vg_via = integrator << viam2m3
vg_via.move(xm1_ref.ports["multiplier_0_gate_S"].center).movey(-1.5)
integrator << straight_route(pdk, xm1_ref.ports["multiplier_0_gate_S"], vg_via.ports["top_met_N"])
integrator << c_route(pdk, xm1_ref.ports["multiplier_0_drain_W"], vg_via.ports["bottom_met_W"])
integrator << L_route(pdk, xm2_ref.ports["multiplier_0_drain_W"], vg_via.ports["top_met_S"])

viam2m31 = via_stack(pdk, "met2", "met3", centered=True)
vg_via1 = integrator << viam2m31
vg_via1.move(xm3_ref.ports["multiplier_0_gate_E"].center).movex(2)
integrator << straight_route(pdk, xm3_ref.ports["multiplier_0_gate_E"], vg_via1.ports["top_met_N"])
integrator << L_route(pdk, xm3_ref.ports["multiplier_0_drain_E"], vg_via1.ports["top_met_N"])
integrator << straight_route(pdk, xm2_ref.ports["multiplier_0_gate_W"], vg_via1.ports["top_met_E"])

viam2m32 = via_stack(pdk, "met2", "met3", centered=True)
vg_via2 = integrator << viam2m32
vg_via2.move(xm1_ref.ports["multiplier_0_source_W"].center).movex(-8)
integrator << L_route(pdk, xm6_ref.ports["multiplier_0_source_E"], vg_via2.ports["top_met_N"])
integrator << straight_route(pdk, xm1_ref.ports["multiplier_0_source_W"], vg_via2.ports["top_met_N"])


integrator << straight_route(pdk, xm2_ref.ports["multiplier_0_source_W"], xm3_ref.ports["multiplier_0_source_E"])

integrator << L_route(pdk, vg_via2.ports["top_met_N"], xc3_ref.ports["row1_col0_top_met_W"])
integrator << straight_route(pdk, xm2_ref.ports["multiplier_0_source_E"], xc3_ref.ports["row0_col0_bottom_met_W"])







ComponentReference (parent Component "straight_route_313ce5b4", ports ['route_W', 'route_N', 'route_E', 'route_S'], origin (0.0, 0.0), rotation 0.0, x_reflection False)

In [7]:
integrator.show()

/headless/conda-env/miniconda3/envs/GLdev/lib/python3.10/site-packages/gdsfactory/show.py:40: UserWarning: Unnamed cells, 5 in 'integrator'
  gdspath = component.write_gds(
2026-08-25 17:18:47.071 | INFO     | gdsfactory.klive:show:55 - Message from klive: {"version": "0.4.1", "klayout_version": "0.30.8", "type": "open", "file": "/tmp/gdsfactory/integrator.gds"}


In [18]:
xc3_ref.ports
#c3_ref.pprint_ports()

In [8]:
# =========================================================================
# 4. ADD AVDD & AVSS POWER RAILS AND CONNECT SUPPLY NODES
# =========================================================================

# https://gemini.google.com/app/131a537106def8cd
# Calculate full horizontal span of the cell to draw supply rails
bbox = evaluate_bbox(integrator) #calcular el tamaño del componente
top_y = integrator.ymax + 3.0
bottom_y = integrator.ymin - 3.0

# Draw horizontal metal2 power rails
avdd_rail = integrator << gf.components.rectangle(
    size=(bbox[0]+4, 1.0), 
    layer=pdk.get_layer("metal2")
)
avdd_rail.move((-2, top_y))

avss_rail = integrator << gf.components.rectangle(
    size=(bbox[0]+4, 1.0), 
    layer=pdk.get_layer("metal2")
)
avss_rail.move((-2, bottom_y))

# Add global ports for the supply rails
integrator.add_port("avdd", center=(bbox[0]/2, top_y + 0.5), width=1.0, orientation=180, layer=pdk.get_layer("metal2"), port_type="electrical")
integrator.add_port("avss", center=(bbox[0]/2, bottom_y + 0.5), width=1.0, orientation=180, layer=pdk.get_layer("metal2"), port_type="electrical")
# add label to those global ports 
integrator.add_label(text="avdd", position=(bbox[0]/2, top_y + 0.5), layer=pdk.get_glayer("met2_label") , magnification=1.5)
integrator.add_label(text="avss", position=(bbox[0]/2, bottom_y + 0.5), layer=pdk.get_glayer("met2_label") , magnification=1.5)

Label(text='avss', origin=(53.98, -4.46), layer=(36, 10))

In [ ]:
xm1_ref.pprint_ports()

In [9]:
integrator << L_route(pdk,xm1_ref.ports["tie_E_top_met_N"], integrator.ports["avdd"])

integrator << L_route(pdk,xm2_ref.ports["tie_E_top_met_S"], integrator.ports["avss"])
integrator << L_route(pdk,xm3_ref.ports["tie_W_top_met_S"], integrator.ports["avss"])
integrator << L_route(pdk,xm6_ref.ports["tie_E_top_met_S"], integrator.ports["avss"])

integrator << L_route(pdk,xm6_ref.ports["multiplier_0_drain_N"], integrator.ports["avdd"])
integrator << straight_route(pdk,xc3_ref.ports["row0_col0_bottom_met_S"], integrator.ports["avss"])

ComponentReference (parent Component "straight_route_b3b5b587", ports ['route_W', 'route_N', 'route_E', 'route_S'], origin (0.0, 0.0), rotation 0.0, x_reflection False)

In [10]:
integrator.show()

2026-08-25 17:21:38.497 | INFO     | gdsfactory.klive:show:55 - Message from klive: {"version": "0.4.1", "klayout_version": "0.30.8", "type": "reload", "file": "/tmp/gdsfactory/integrator.gds"}


In [11]:
integrator.name="integrator"
drc_result = gf180.drc_magic(integrator, integrator.name)

/headless/conda-env/miniconda3/envs/GLdev/lib/python3.10/site-packages/glayout/pdk/mappedpdk.py:540: UserWarning: Unnamed cells, 5 in 'integrator'
  layout.write_gds(gds_path)
2026-08-25 17:22:07.825 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/tmp/tmp561u5g5n/integrator.gds'


using default pdk_root
Defaulting to stale magic_commands.tcl

Magic 8.3 revision 636 - Compiled on Thu Apr 16 09:13:57 PM CEST 2026.
Starting magic under Tcl interpreter
Using the terminal as the console.
Using NULL graphics device.
Processing system .magicrc file
Switching to WIRING tool.
Switching to NETLIST tool.
Switching to PICK tool.
Switching to BOX tool.
Sourcing design .magicrc for technology gf180mcuD ...
10 Magic internal units = 1 Lambda
Input style import: scaleFactor=10, multiplier=2
The following types are not handled by extraction and will be treated as non-electrical types:
    obsactive mvobsactive filldiff fillpoly m1hole obsm1 fillm1 obsv1 m2hole obsm2 fillm2 obsv2 m3hole obsm3 fillm3 m4hole obsm4 fillm4 m5hole obsm5 fillm5 glass fillblock lvstext obscomment 
Scaled tech values by 10 / 1 to match internal grid scaling
Loading gf180mcuD Device Generator Menu ...
Loading "/tmp/tmp561u5g5n/magic_commands.tcl" from command line.
Library written using GDS-II Release 6.0

In [12]:
import os
from pathlib import Path
import tempfile
magicrc_file = Path(os.environ['PDKPATH']) / "libs.tech" / "magic" / f"{os.environ['PDK']}.magicrc"
design_name=integrator.name
path_to_dir = "/foss/designs/designs/stdp/"

pex_path = path_to_dir + f"{design_name}.spice"
gds_path = path_to_dir + f"{design_name}.gds"

integrator.write_gds(str(gds_path))
    
# magic_script_content = f"""
# drc off            
# gds flatglob *\\$\\$*
# gds read {gds_path}

# flatten {design_name}
# load {design_name}
# select top cell
# extract do local
# extract all
# ext2sim labels on
# ext2sim
# extresist tolerance 10
# extresist
# ext2spice lvs
# ext2spice cthresh 0
# ext2spice extresist on
# ext2spice -o {str(pex_path)}
# exit
# """

magic_script_content = f"""
drc off            
gds flatglob *\\$\\$*
gds read {gds_path}
load {design_name}
select top cell
extract do local
extract all
ext2sim labels on
ext2sim
ext2spice lvs
ext2spice -o {str(pex_path)}
exit
"""

with tempfile.NamedTemporaryFile(mode='w', delete=False) as magic_script_file:
    magic_script_file.write(magic_script_content)
    magic_script_path = magic_script_file.name
    
magic_cmd = f"bash -c 'magic -rcfile {magicrc_file} -noconsole -dnull < {magic_script_path}'",
magic_subproc = subprocess.run(
    magic_cmd, 
    shell=True,
    check=True,
    capture_output=True
)

magic_subproc_code = magic_subproc.returncode
magic_subproc_out = magic_subproc.stdout.decode('utf-8')
print(magic_subproc_out)

/tmp/ipykernel_423/860907566.py:11: UserWarning: Unnamed cells, 5 in 'integrator'
  integrator.write_gds(str(gds_path))
2026-08-25 17:22:35.006 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/foss/designs/designs/stdp/integrator.gds'



Magic 8.3 revision 636 - Compiled on Thu Apr 16 09:13:57 PM CEST 2026.
Starting magic under Tcl interpreter
Using the terminal as the console.
Using NULL graphics device.
Processing system .magicrc file
Switching to WIRING tool.
Switching to NETLIST tool.
Switching to PICK tool.
Switching to BOX tool.
Sourcing design .magicrc for technology gf180mcuD ...
10 Magic internal units = 1 Lambda
Input style import: scaleFactor=10, multiplier=2
The following types are not handled by extraction and will be treated as non-electrical types:
    obsactive mvobsactive filldiff fillpoly m1hole obsm1 fillm1 obsv1 m2hole obsm2 fillm2 obsv2 m3hole obsm3 fillm3 m4hole obsm4 fillm4 m5hole obsm5 fillm5 glass fillblock lvstext obscomment 
Scaled tech values by 10 / 1 to match internal grid scaling
Loading gf180mcuD Device Generator Menu ...
Using technology "gf180mcuD", version 1.0.575-2-g7b70722
Library written using GDS-II Release 6.0
Library name: library
Reading "integrator".
Reading "Unnamed_d3c665ed

In [13]:
import glob
extensions = [
            "els"
            "*.gds",
            "*.ext",
            "*.res.ext",
            "*.lvs.rpt",
            "*_lvs.rpt",
            "*.nodes",
            "*.sim",
            "*.pex.spice",
            "*_pex.spice"
            ]
files_to_delete = []
for ext in extensions:
    files_to_delete.extend(glob.glob(ext))
    
# Delete the files
for file_path in files_to_delete:
    try:
        os.remove(file_path)
        print(f"Deleted: {file_path}")
    except OSError as e:
        print(f"Error deleting {file_path}: {e}")

In [14]:
import os
from pathlib import Path

gf180.lvs_netgen(
    layout=integrator,
    design_name = integrator.name,
    pdk_root = Path(os.environ['PDKPATH']),
    lvs_setup_tcl_file = Path(os.environ['PDKPATH']) / "libs.tech" / "netgen" / f"{os.environ['PDK']}_setup.tcl",
    lvs_schematic_ref_file = Path(str(path_to_dir + "integrator.spice")),
    netlist = Path(str(path_to_dir + "integrator_sch.spice")),
    output_file_path =  Path(str(path_to_dir))
)

/headless/conda-env/miniconda3/envs/GLdev/lib/python3.10/site-packages/glayout/pdk/mappedpdk.py:762: UserWarning: Unnamed cells, 5 in 'integrator'
  layout.write_gds(str(gds_path))
2026-08-25 17:23:06.129 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/tmp/tmpnstq36eu/integrator.gds'


using user specified pdk_root, will search for required files in the specified directory

Magic 8.3 revision 636 - Compiled on Thu Apr 16 09:13:57 PM CEST 2026.
Starting magic under Tcl interpreter
Using the terminal as the console.
Using NULL graphics device.
Processing system .magicrc file
Switching to WIRING tool.
Switching to NETLIST tool.
Switching to PICK tool.
Switching to BOX tool.
Sourcing design .magicrc for technology gf180mcuD ...
10 Magic internal units = 1 Lambda
Input style import: scaleFactor=10, multiplier=2
The following types are not handled by extraction and will be treated as non-electrical types:
    obsactive mvobsactive filldiff fillpoly m1hole obsm1 fillm1 obsv1 m2hole obsm2 fillm2 obsv2 m3hole obsm3 fillm3 m4hole obsm4 fillm4 m5hole obsm5 fillm5 glass fillblock lvstext obscomment 
Scaled tech values by 10 / 1 to match internal grid scaling
Loading gf180mcuD Device Generator Menu ...
Using technology "gf180mcuD", version 1.0.575-2-g7b70722
Library written using

{'magic_subproc_code': 0,
 'netgen_subproc_code': 0,
 'result_str': 'LVS run succeeded\nErrors found in LVS report: /tmp/tmpnstq36eu/integrator_lvs.rpt'}

In [11]:

# =========================================================================
# 6. PHYSICAL PORTS & LVS LABELS FOR SIGNALS
# =========================================================================
# Asignación de etiquetas LVS en la capa met2_label para señales de entrada/control
integrator.add_label(text="vext", position=xm6_ref.ports["multiplier_0_gate_E"].center, layer=pdk.get_glayer("met2_label"), magnification=1.0)
integrator.add_label(text="i_50n", position=xm3_ref.ports["multiplier_0_gate_E"].center, layer=pdk.get_glayer("met2_label"), magnification=1.0)

# Asegurar nombre de celda
integrator.name = "integrator"


# =========================================================================
# 7. SCHEMATIC NETLIST DEFINITION (FOR LVS)
# =========================================================================
netlist = Netlist(
    circuit_name="integrator",
    nodes=['avss', 'avdd', 'vext', 'i_50n', 'vx', 'vy', 'vout']
)

# Transistores
# XM1 (PMOS): D=vx, G=vx, S=avdd, B=avdd
netlist.connect_netlist(xm1.info['netlist'], [('D', 'vx'), ('G', 'vx'), ('S', 'avdd'), ('B', 'avdd')])

# XM6 (NMOS): D=vy, G=vext, S=avss, B=avss
netlist.connect_netlist(xm6.info['netlist'], [('D', 'vy'), ('G', 'vext'), ('S', 'avss'), ('B', 'avss')])

# XM3 (NMOS): D=i_50n, G=i_50n, S=avss, B=avss
netlist.connect_netlist(xm3.info['netlist'], [('D', 'i_50n'), ('G', 'i_50n'), ('S', 'avss'), ('B', 'avss')])

# XM2 (NMOS): D=vx, G=i_50n, S=vout, B=avss
netlist.connect_netlist(xm2.info['netlist'], [('D', 'vx'), ('G', 'i_50n'), ('S', 'vout'), ('B', 'avss')])

# Arreglo Capacitor XC3: V1=vy, V2=vout
netlist.connect_netlist(xc3.info["netlist"], [("V1", "vy"), ("V2", "vout")])

# Asignar netlist al componente
integrator.info["netlist"] = netlist


# =========================================================================
# 8. PRINT NETLIST (CDL / SPICE FORMAT)
# =========================================================================
print("=== NETLIST ESQUEMÁTICA GENERADA ===")
print(integrator.info['netlist'].generate_netlist())


# =========================================================================
# 9. DRC VERIFICATION VIA MAGIC VLSI & EXPORT
# =========================================================================
output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)
gds_file = os.path.join(output_dir, "integrator.gds")

# Guardar GDSII
integrator.write_gds(gds_file)

# Ejecutar DRC con Magic VLSI
try:
    drc_result = gf180.drc_magic(integrator, integrator.name)
    print("\n=== REPORTE DRC (MAGIC VLSI) ===")
    print(drc_result)
except Exception as e:
    print(f"\nNota/Advertencia al ejecutar DRC: {e}")

# Visualizar layout en Jupyter si la función display_component está disponible
try:
    display_component(integrator, output_dir, scale=3)
except Exception:
    pass

=== NETLIST ESQUEMÁTICA GENERADA ===
.subckt PMOS D G S B DUM l=0.28 w=1.0
XMAIN1 D G S B pfet_03v3 l=0.28 w=1.0
.ends PMOS

.subckt NMOS D G S B DUM l=0.28 w=1.0
XMAIN1 D G S B nfet_03v3 l=0.28 w=1.0
.ends NMOS

.subckt MIMCap V1 V2 l=1 w=1
X1 V1 V2 mimcap_1p0fF l={l} w={w}
.ends MIMCap

.subckt MIMCAP_ARR V1 V2
X0 V1 V2 MIMCap l=17.68 w=17.68
X1 V1 V2 MIMCap l=17.68 w=17.68
X2 V1 V2 MIMCap l=17.68 w=17.68
X3 V1 V2 MIMCap l=17.68 w=17.68
X4 V1 V2 MIMCap l=17.68 w=17.68
X5 V1 V2 MIMCap l=17.68 w=17.68
X6 V1 V2 MIMCap l=17.68 w=17.68
X7 V1 V2 MIMCap l=17.68 w=17.68
.ends MIMCAP_ARR

.subckt integrator avss avdd vext i_50n vx vy vout
X0 vx vx avdd avdd DUM PMOS l=0.28 w=1.0
X1 vy vext avss avss DUM NMOS l=0.28 w=1.0
X2 i_50n i_50n avss avss DUM NMOS l=0.28 w=1.0
X3 vx i_50n vout avss DUM NMOS l=0.28 w=1.0
X4 vy vout MIMCAP_ARR
.ends integrator


PermissionError: [Errno 13] Permission denied: './output'